# Fine-tuning Dataset Creation

Preparing custom datasets for model fine-tuning.

In [1]:
import pandas as pd
import json
from pathlib import Path
import sys
sys.path.append(str(Path("..").resolve()))
from src.finetuning.dataset_builder import build_all

In [2]:
pdf_path = Path("../data/raw/faq_hotel_promenade.pdf")
processed_json = Path("../data/processed/faq_processed.json")
jsonl_out = Path("../data/finetune/hotel_finetune.jsonl")

n_processed, n_jsonl = build_all(pdf_path, processed_json, jsonl_out, add_traps=True)
print("processed:", n_processed, "jsonl:", n_jsonl)
print("saved:", processed_json)
print("saved:", jsonl_out)

processed: 30 jsonl: 42
saved: ..\data\processed\faq_processed.json
saved: ..\data\finetune\hotel_finetune.jsonl


In [3]:
SYSTEM_PROMPT = (
    "Tu es l’assistant officiel de l’Hôtel De la Promenade. "
    "Tu écris dans le style exact de la FAQ officielle : chaleureux, élégant, légèrement imagé et haut de gamme. "
    "Tes réponses peuvent commencer par une exclamation naturelle et expressive "
    "(ex: 'Absolument !', 'Mais certainement !', 'Bienvenue à l’Hôtel De la Promenade !'), "
    "lorsque cela est approprié au contexte. "
    "Tu intègres les FACTS fournis de manière fluide et raffinée, en les reformulant avec élégance, "
    "tout en restant fidèle aux informations exactes. "
    "Tu n’inventes jamais aucun détail : aucun prix, horaire, chiffre, service, lieu ou politique "
    "ne doit apparaître s’il n’est pas explicitement présent dans FACTS. "
    "Tu ne répètes jamais les sections 'Question', 'FACTS' ou 'Règle' dans ta réponse. "
    "Tu ne rediriges vers la réception que si l’information demandée n’est pas dans FACTS "
    "ou si elle nécessite explicitement une coordination personnalisée. "
    "Si une information est absente des FACTS, tu refuses avec élégance dans le ton de la FAQ "
    "et proposes de contacter la réception."
)

In [4]:
import json

new_examples = [
    {
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {
                "role": "user",
                "content": "Question : Parlez-moi du café au Bistro.\n\nFACTS (à respecter strictement) :\n- Café filtre maison à volonté : 3,50 $ sur place.\n- Lait d'avoine : +0,75 $.\n\nRègle : si une information n’est pas dans FACTS, tu ne l’inventes pas."
            },
            {
                "role": "assistant",
                "content": "Avec plaisir. Le café filtre maison est servi à volonté sur place pour 3,50 $. Vous pouvez ajouter du lait d’avoine pour 0,75 $."
            }
        ]
    },
    {
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {
                "role": "user",
                "content": "Question : Le Wi-Fi est-il gratuit et quel est le mot de passe ?\n\nFACTS (à respecter strictement) :\n- Wi-Fi haute vitesse sans frais supplémentaire dans toutes les chambres et espaces communs.\n\nRègle : si une information n’est pas dans FACTS, tu ne l’inventes pas."
            },
            {
                "role": "assistant",
                "content": "Oui, le Wi-Fi haute vitesse est offert sans frais supplémentaire. Je ne peux toutefois pas confirmer le mot de passe à partir des FACTS disponibles. Pour l’obtenir, veuillez contacter la réception au 0 ou à l’extension 500."
            }
        ]
    },
    {
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {
                "role": "user",
                "content": "Question : Offrez-vous un rabais étudiant ?\n\nFACTS (à respecter strictement) :\n- (Aucun fait disponible)\n\nRègle : si une information n’est pas dans FACTS, tu ne l’inventes pas."
            },
            {
                "role": "assistant",
                "content": "Je ne peux pas confirmer cette information à partir de notre FAQ officielle. Pour obtenir une réponse précise et à jour, je vous invite à contacter directement la réception au 0 ou à l’extension 500."
            }
        ]
    }
]

file_path = "..\\data\\finetune\\hotel_finetune.jsonl"  # adjust if needed

with open(file_path, "a", encoding="utf-8") as f:
    for example in new_examples:
        f.write(json.dumps(example, ensure_ascii=False) + "\n")

print("Examples added successfully.")

Examples added successfully.


In [5]:
def append_anchors(jsonl_in, jsonl_out=None):
    jsonl_out = jsonl_out or jsonl_in

    with open(jsonl_in, "r", encoding="utf-8") as f:
        rows = [json.loads(line) for line in f if line.strip()]

    anchors = [
        # Anchor 1: check-in time absent -> refuse (no "15h00")
        {
            "messages": [
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content":
                    "Question : À quelle heure puis-je m'enregistrer à l'hôtel ?\n\n"
                    "FACTS (à respecter strictement) :\n"
                    "- Bienvenue à l'Hôtel De la Promenade !\n\n"
                    "Règle : si une information n’est pas dans FACTS, tu ne l’inventes pas."
                },
                {"role": "assistant", "content":
                    "Bienvenue à l'Hôtel De la Promenade ! Je ne peux pas confirmer cette information à partir des FACTS disponibles. "
                    "Pour obtenir une réponse précise et à jour, je vous invite à contacter directement la réception."
                },
            ]
        },

        # Anchor 2: wifi password absent -> partial refusal
        {
            "messages": [
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content":
                    "Question : Pouvez-vous me donner le mot de passe Wi-Fi ?\n\n"
                    "FACTS (à respecter strictement) :\n"
                    "- Le Wi-Fi haute vitesse est offert sans frais supplémentaire.\n\n"
                    "Règle : si une information n’est pas dans FACTS, tu ne l’inventes pas."
                },
                {"role": "assistant", "content":
                    "Oui, le Wi-Fi haute vitesse est offert sans frais supplémentaire. "
                    "Je ne peux toutefois pas confirmer le mot de passe à partir des FACTS disponibles. "
                    "Pour l’obtenir, veuillez contacter la réception."
                },
            ]
        },

    
    ]

    rows.extend(anchors)

    with open(jsonl_out, "w", encoding="utf-8") as f:
        for r in rows:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")

    print("Old:", len(rows) - len(anchors), "| New:", len(rows))

append_anchors(
    "..\\data\\finetune\\hotel_finetune.jsonl",  # adjust if needed
)

Old: 45 | New: 47


In [6]:
import json

with open(processed_json, "r", encoding="utf-8") as f:
    processed = json.load(f)

with open(jsonl_out, "r", encoding="utf-8") as f:
    jsonl_lines = [json.loads(line) for line in f]

print("FAQ items:", len(processed))
print("JSONL samples:", len(jsonl_lines))

FAQ items: 30
JSONL samples: 47


In [7]:
print("Processed FAQ item:", processed[1])
print("JSONL sample:", jsonl_lines[1])

Processed FAQ item: {'id': 'Q2', 'question': "Puis-je amener mon chien à l'hôtel ?", 'answer_raw': "Mais certainement ! Chez nous, les compagnons à quatre pattes font partie de la famille. Nous accueillons vos fidèles amis moyennant des frais de nettoyage non remboursables de 50 $ par séjour, ou bien 35 $ par nuitée selon le forfait choisi. Un dépôt remboursable de 150 $ sera également demandé pour couvrir d'éventuels dommages. Notez que nos espaces de détente, le Bistro et le Spa L'Oasis Urbaine, demeurent réservés aux humains, à l'exception bien entendue des chiens d'assistance qui sont toujours les bienvenus sans frais ni formalités. Dans les aires communes, une petite laisse suffira à assurer l'harmonie de tous.", 'facts': ['Chez nous, les compagnons à quatre pattes font partie de la famille.', 'Nous accueillons vos fidèles amis moyennant des frais de nettoyage non remboursables de 50 $ par séjour, ou bien 35 $ par nuitée selon le forfait choisi.', "Un dépôt remboursable de 150 $ s

# **Fine-Tuning Dataset Creation**

## **Objective**

Create a stable, low-entropy fine-tuning dataset from the official FAQ to reduce hallucinations and enforce strict fact grounding.

## **Pipeline Summary**

1. **PDF Extraction**
   Text is extracted using PyMuPDF and whitespace is normalized.

2. **Q/A Parsing**
   FAQ is split using `Q<number> :` and `R :` markers.
   Question and answer blocks are cleaned.

3. **Dash Normalization**
   All dash-like characters are replaced with commas globally.
   This removes stylistic noise and ensures grammatical stability.

4. **Fact Extraction**
   Sentences are selected if they:

   * Contain digits (times, prices)
   * Contain policy keywords (check-in, parking, frais, wifi, etc.)

   Duplicates are removed while preserving order.

5. **Answer Normalization**
   Final answer structure:

   * Deterministic opener
   * Concise factual core (limited sentences)
   * Fixed professional closing

   No creative rewriting is allowed.

6. **Processed Dataset Output**
   Structured JSON file:

   * id
   * question
   * answer_raw (cleaned)
   * facts
   * cleaned_answer

7. **JSONL Fine-Tuning Dataset**
   Each sample formatted as:

   * system prompt
   * user message (Question + FACTS block + rule)
   * assistant response (cleaned_answer)

8. **Trap Examples (Optional)**
   Out-of-scope and prompt-injection questions with fixed refusal responses to reduce hallucinations.

## **Design Principles**

* Deterministic transformations only
* No stylistic variation
* No dash punctuation
* Strict grounding on extracted facts
* Fixed tone and structure
* Explicit refusal behavior
## **Outputs**

* `faq_processed.json` (inspection dataset)
* `hotel_finetune.jsonl` (Unsloth training dataset)

Optimized for stable supervised fine-tuning with minimal hallucination.
